In [26]:
from backtesting import Backtest, Strategy
import pandas as pd
from backtesting.lib import crossover, plot_heatmaps, resample_apply
import seaborn as sns
import matplotlib.pyplot as plt
import mpld3
import numpy as np
import time
from talib import CDLENGULFING, ADX, CCI, ATR
from DataPaths import data_paths
import plotly.express as px
from concurrent.futures import ThreadPoolExecutor, as_completed
from backtesting import Backtest
import pandas as pd
import plotly.express as px

In [27]:
def load_and_prepare_data(file_path, start_date, end_date):
    # Load CSV
    data = pd.read_csv(file_path, parse_dates=['Time'], index_col='Time')
    # Ensure index is datetime
    data.index = pd.to_datetime(data.index)

    # Print column names to verify
    # print("Columns in the CSV file:", data.columns)

    # Select required columns
    data = data[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

    # Remove duplicate indexes
    if data.index.duplicated().any():
        print("Duplicate indexes found. Removing duplicates.")
        data = data[~data.index.duplicated(keep='first')]

    # # Check for NaN values
    # print("Checking for NaN values in the data:")
    # print(data.isna().sum())

    # Drop NaN values
    data = data.dropna()

    # Reduce the number of data points to a specific date range
    data = data.loc[start_date:end_date]

    # print(f"Number of data points after reduction: {len(data)}")

    return data


In [28]:
class SimpleEngulfingStrategy(Strategy):
    SL = 50
    TP_R = 4
    days_held = 1
    MAX_HOLD_TIME = pd.Timedelta(days=days_held)

    def init(self):
        self.engulfing = self.I(CDLENGULFING, self.data.Open, self.data.High, self.data.Low, self.data.Close)
        self.entry_price = None
        self.stop_loss = None
        self.take_profit = None
        self.entry_time = None

    def next(self):
        engulf = self.engulfing[-1]
        price = self.data.Close[-1]
        current_time = self.data.index[-1]

        if current_time.weekday() in [0, 3]:
            return

        if not self.position:
            if engulf == 100:
                self.buy()
                self.entry_price = price
                self.stop_loss = price - self.SL
                self.take_profit = price + self.SL * self.TP_R
                self.entry_time = current_time

            elif engulf == -100:
                self.sell()
                self.entry_price = price
                self.stop_loss = price + self.SL
                self.take_profit = price - self.SL * self.TP_R
                self.entry_time = current_time

        if self.position:
            if self.position.is_long:
                if (
                    price <= self.stop_loss
                    or price >= self.take_profit
                    or (current_time - self.entry_time) > self.MAX_HOLD_TIME
                ):
                    self.position.close()
                    self.entry_price = None
                    self.stop_loss = None
                    self.take_profit = None
                    self.entry_time = None

            elif self.position.is_short:
                if (
                    price >= self.stop_loss
                    or price <= self.take_profit
                    or (current_time - self.entry_time) > self.MAX_HOLD_TIME
                ):
                    self.position.close()
                    self.entry_price = None
                    self.stop_loss = None
                    self.take_profit = None
                    self.entry_time = None


In [29]:
def run_backtest(data, strategy, cash, commission):
    """
    Runs a single backtest and returns the results.
    """
    bt = Backtest(data, strategy, cash=cash, commission=commission)
    stats = bt.run()
    return stats.get("Return [%]", None)

# Function to run multiple backtests concurrently
def multi_threaded_backtesting(data_sets, strategy, cash=10_000_000, commission=0.002):
    """
    Runs multiple backtests concurrently using ThreadPoolExecutor.
    """
    results = {}
    with ThreadPoolExecutor(max_workers=8) as executor:
        future_to_label = {
            executor.submit(run_backtest, data, strategy, cash, commission): label
            for data, label in data_sets
        }
        for future in as_completed(future_to_label):
            label = future_to_label[future]
            try:
                results[label] = future.result()
            except Exception as e:
                print(f"Error in backtest for {label}: {e}")
                results[label] = None
    return results


In [ ]:
print('working...')
if __name__ == '__main__':
    # Example datasets
    data_paths = {
        "GBPUSD_M15": data_paths["GBPUSD"]["M15"],
        "US30_M15": data_paths["US30"]["M15"],
        "NASDAQ_M15": data_paths["NAS100"]["M15"],
        "BTC_M15": data_paths["BTC"]["M15"],
    }

    start_date = '2024-01-01'
    end_date = '2024-07-01'

    # Load datasets
    data_sets = []
    for label, file_path in data_paths.items():
        df = load_and_prepare_data(file_path, start_date, end_date)
        data_sets.append((df, label))

    # Run multi-threaded backtesting
    results = multi_threaded_backtesting(data_sets, SimpleEngulfingStrategy)

    # Print results
    for label, ret in results.items():
        print(f"{label}: {ret}%")

    # Plot results
    labels = list(results.keys())
    returns = list(results.values())
    fig = px.scatter(x=labels, y=returns, labels={"x": "Dataset", "y": "Returns (%)"}, title="Backtest Results")
    fig.show()

working...
NASDAQ_M15: -53.72335264052008%
US30_M15: -58.67527134246009%
GBPUSD_M15: -27.523930108493992%
BTC_M15: -94.876271596%
